# Permuting and sampling data to calculate the null hypothesis

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
data_loc = '../data/server_ready'

files = [os.path.join(data_loc, f) for f in os.listdir(data_loc) if (not f.startswith('._')) and f.endswith('.tsv') and ('null-corpus' not in f)]
print(files)

['../data/server_ready/cha_corpus-callfriend.tsv', '../data/server_ready/cha_corpus-callhome.tsv', '../data/server_ready/callfriend-ko_corpus.tsv', '../data/server_ready/cha_corpus-yiddish.tsv', '../data/server_ready/d_cha_corpus-croatian.tsv', '../data/server_ready/finchat_corpus.tsv']


In [3]:
null_output_file = 'null-corpora/{}-null-corpus.parquet'
null_output_file = os.path.join(data_loc, null_output_file)
null_output_file

'../data/server_ready/null-corpora/{}-null-corpus.parquet'

In [4]:
def lang(x):
    return x.split('-')[1]

## Setting up the null-test data

### Creating language specific corpora

There are multiple languages spread across multiple corpora. Given that it's kinda cheating to permute sentences from one language to compare to sentences in another we need to start by aggregating data for each language.

In [5]:
df_ = dict()

In [6]:
for f in tqdm(files):
    print(f)
    # df = pd.read_csv(f)
    df = pd.read_table(f, sep='\t')

    # print(df[['x_utterance', 'y_utterance']].isna().sum(),'\n')
    df = df.loc[
        (~df['x_utterance'].isna())
        & (~df['y_utterance'].isna())
    ]
    # print(df[['x_utterance', 'y_utterance']].isna().sum())

    df['x_turn_id'] = df['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)

    df['sample_delta'] = df.groupby(by=['x_turn_id', 'self']).cumcount() + 1

    print('capturing language labels')
    df['lang'] = [lang(x) for x in tqdm(df['x_speaker'])]
    print(df['lang'].unique())

    for l in df['lang'].unique():
        if l not in df_.keys():
            df_[l] = [df.loc[df['lang'].isin([l])].copy()]
        else:
            df_[l] += [df.loc[df['lang'].isin([l])].copy()]

    del df

  0%|          | 0/6 [00:00<?, ?it/s]

../data/server_ready/cha_corpus-callfriend.tsv
capturing language labels


  0%|          | 0/11049978 [00:00<?, ?it/s]

['eng' 'fra' 'spa' 'zho']
../data/server_ready/cha_corpus-callhome.tsv
capturing language labels


  0%|          | 0/11122234 [00:00<?, ?it/s]

...

['yid']
../data/server_ready/d_cha_corpus-croatian.tsv
capturing language labels


  0%|          | 0/3315889 [00:00<?, ?it/s]

['cro']
../data/server_ready/finchat_corpus.tsv
capturing language labels


  0%|          | 0/92605 [00:00<?, ?it/s]

['fin']


In [7]:
for language,docs in tqdm(df_.items()):
    print(language)

    df = pd.concat(docs, ignore_index=True)
    df['null_y_utterance'] = df['y_utterance'].sample(frac=1).reset_index(drop=True)

    bad = df['null_y_utterance'] == df['y_utterance']
    print(bad.sum(), bad.mean())

    df.to_parquet(
        str(null_output_file).format(language).replace('.tsv', '.parquet')
    )

    del df
    # del df_[language]

  0%|          | 0/10 [00:00<?, ?it/s]

eng
18010 0.004084003947492544
fra
5314 0.0022569950281550997
spa
26854 0.003333335815895198
zho
8488 0.002688950403310881
deu
20831 0.01079200611327695
jpn
93701 0.04137229145739737
ko
48233 0.02195114070110941
yid
31 0.002362624799939029
cro
3788 0.00114237840892744
fin
72 0.0007774958155607149


In [8]:
del df_

### Load, permute, save

In [5]:
files = [os.path.join(data_loc, 'null-corpora', f) for f in os.listdir(os.path.join(data_loc, 'null-corpora')) if (not f.startswith('._')) and f.endswith('.parquet')]
files

['../data/server_ready/null-corpora/eng-null-corpus.parquet',
 '../data/server_ready/null-corpora/fra-null-corpus.parquet',
 '../data/server_ready/null-corpora/spa-null-corpus.parquet',
 '../data/server_ready/null-corpora/zho-null-corpus.parquet',
 '../data/server_ready/null-corpora/deu-null-corpus.parquet',
 '../data/server_ready/null-corpora/jpn-null-corpus.parquet',
 '../data/server_ready/null-corpora/ko-null-corpus.parquet',
 '../data/server_ready/null-corpora/yid-null-corpus.parquet',
 '../data/server_ready/null-corpora/cro-null-corpus.parquet',
 '../data/server_ready/null-corpora/fin-null-corpus.parquet']

In [6]:
for f in tqdm(files):
    print(f)
    # df = pd.read_csv(f)
    # df = pd.read_table(f, sep='\t')
    df = pd.read_parquet(f)

    # print(df[['x_utterance', 'y_utterance']].isna().sum(),'\n')
    df = df.loc[
        (~df['x_utterance'].isna())
        & (~df['y_utterance'].isna())
    ]
    # print(df[['x_utterance', 'y_utterance']].isna().sum())

    print('finding examples')
    # a lot of work just to find useful examples ...
    good_examples = df[['corpus', 'x_turn_id', 'self', 'sample_delta']].groupby(by=['corpus', 'x_turn_id', 'self', 'sample_delta']).agg('max').reset_index()
    good_examples = pd.merge(
        left=good_examples.loc[good_examples['self']], left_on='x_turn_id',
        right=good_examples[['x_turn_id', 'self', 'sample_delta']].loc[~good_examples['self']], right_on='x_turn_id',
        how='left'
    )
    good_examples = good_examples.loc[
        ((good_examples['sample_delta_x'] >= 5)
        & (good_examples['sample_delta_x'] <= 70))
        & ((good_examples['sample_delta_y'] >= 5)
        & (good_examples['sample_delta_y'] <= 70))
    ]

    df = df.loc[df['x_turn_id'].isin(good_examples['x_turn_id'])]
    # print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    print('saving data')
    sample_ids = []

    for c in tqdm(df['corpus'].unique()):

        sub = df['x_turn_id'].loc[df['corpus'].isin([c])].unique()

        k = int(np.ceil(len(sub)*.1))

        if k > 0:
            sample_ids += np.random.choice(sub, size=(k,), replace=False).tolist()

    df = df.loc[df['x_turn_id'].isin(sample_ids)]
    df.to_parquet(f)

    print('===][===')

  0%|          | 0/10 [00:00<?, ?it/s]

../data/server_ready/null-corpora/eng-null-corpus.parquet
finding examples
saving data


  0%|          | 0/3 [00:00<?, ?it/s]

===][===
../data/server_ready/null-corpora/fra-null-corpus.parquet
finding examples
saving data


  0%|          | 0/1 [00:00<?, ?it/s]

...

===][===
../data/server_ready/null-corpora/cro-null-corpus.parquet
finding examples
saving data


  0%|          | 0/1 [00:00<?, ?it/s]

===][===
../data/server_ready/null-corpora/fin-null-corpus.parquet
finding examples
saving data


  0%|          | 0/1 [00:00<?, ?it/s]

===][===


## Compile null document

In [5]:
corpora_are_in = os.path.join(data_loc, 'null-corpora')
corpora_are_in

'../data/server_ready/null-corpora'

In [6]:
corpora = [f for f in os.listdir(corpora_are_in) if (not f.startswith('._')) and f.endswith('.parquet')]
corpora

['eng-null-corpus.parquet',
 'fra-null-corpus.parquet',
 'spa-null-corpus.parquet',
 'zho-null-corpus.parquet',
 'deu-null-corpus.parquet',
 'jpn-null-corpus.parquet',
 'ko-null-corpus.parquet',
 'yid-null-corpus.parquet',
 'cro-null-corpus.parquet',
 'fin-null-corpus.parquet']

In [7]:
df = []

In [8]:
for f in tqdm(corpora):
    # df += [pd.read_table(os.path.join(corpora_are_in,f), sep='\t')]
    df += [pd.read_parquet(os.path.join(corpora_are_in,f))]
    # df[-1]['y_utterance'] = df[-1]['null_y_utterance'].values
    # del df[-1]['null_y_utterance']
    # print(f, df[-1].shape)
    # print(df[-1][['x_utterance', 'y_utterance']].isna().sum())
    # print('===][===\n')
df = pd.concat(df, ignore_index=True)

  0%|          | 0/10 [00:00<?, ?it/s]

In [9]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [10]:
sel = (df['y_utterance'] == df['null_y_utterance']) & (~df['lang'].isin(['jpn']))

In [11]:
k = df['x_turn_id'].loc[sel].value_counts()

In [12]:
# k[k>1]

In [13]:
df[['x_turn_id', 'y_utterance', 'null_y_utterance']].loc[sel].head(n=100)

,x_turn_id,y_utterance,null_y_utterance
187,callfriend-eng-n-4175-55,right,right
672,callfriend-eng-n-4175-97,hhh hhh hhh hhh,hhh hhh hhh hhh
677,callfriend-eng-n-4175-97,hhh,hhh
1176,callfriend-eng-n-4175-185,yeah,yeah
1219,callfriend-eng-n-4175-197,m hm,m hm
...,...,...,...
31387,callfriend-eng-n-5000-740,yeah,yeah
32121,callfriend-eng-n-5000-942,hhh,hhh
32505,callfriend-eng-n-5000-1005,yeah,yeah
33038,callfriend-eng-n-5000-1087,yeah,yeah


In [14]:
df['y_utterance'] = df['null_y_utterance'].values
del df['null_y_utterance']

In [15]:
null_output_file = os.path.join(data_loc,'null-corpus.tsv')

In [16]:
df.to_csv(null_output_file, index=False, encoding='utf-8', sep='\t')

## Test that stitched doc is okay

In [17]:
null_output_file = os.path.join(data_loc,'null-corpus.tsv')

In [18]:
# df = pd.read_csv(null_output_file)
df = pd.read_table(null_output_file, sep='\t')
df.shape

(2560745, 28)

In [19]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [20]:
df.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,recipient,overlapping_text,corpus,conversation_id,com,...,mor,gra,exp,time,err,sit,add,act,wor,TIME
0,59.0,callfriend-eng-n-4175-26,callfriend-eng-n-callfriend-eng-n-4175-M1,voice number blah blo blo blah,"[49100, 50565]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,59.0,callfriend-eng-n-4175-26,callfriend-eng-n-callfriend-eng-n-4175-M1,voice number blah blo blo blah,"[49100, 50565]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,59.0,callfriend-eng-n-4175-26,callfriend-eng-n-callfriend-eng-n-4175-M1,voice number blah blo blo blah,"[49100, 50565]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,59.0,callfriend-eng-n-4175-26,callfriend-eng-n-callfriend-eng-n-4175-M1,voice number blah blo blo blah,"[49100, 50565]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,59.0,callfriend-eng-n-4175-26,callfriend-eng-n-callfriend-eng-n-4175-M1,voice number blah blo blo blah,"[49100, 50565]",ADULT,True,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
